In [26]:
import torch 
import torch.nn as nn
from torch.utils.data import DataLoader

import pandas as pd
import numpy as np

In [27]:
live = pd.read_csv("../data/samples/trial/live_metrics.csv")
verbose = pd.read_csv("../data/samples/trial/verbose_statements.csv")
initial = pd.read_csv("../data/samples/trial/initial_statement.csv")

In [28]:
live = live.drop(['Unnamed: 0', "_id", "Category", "Collect", ], axis=1)
live = live.dropna()

In [29]:
live['Current Time'] = pd.to_datetime(live['Current Time'])
live['Current Time'] = pd.to_numeric(live['Current Time'])

In [30]:
response_time = verbose['total_duration'].tolist()
response_time = response_time[:-1]
reset_iters = live[live['Iteration'] == 1].index.tolist()

In [31]:
live = live.drop(columns=['Memory Clock Utilization', 'Memory Current Clock (MHz)'])

In [32]:
column_names = live.columns.tolist()
live_avg = dict(zip(column_names, [list(range(70))] * len(column_names)))
live_avg = pd.DataFrame(live_avg, dtype=np.float64)

In [33]:
for iter in range(1, len(reset_iters)):
    for column in live.columns:
        temp_list = live.iloc[reset_iters[iter-1]:reset_iters[iter],  column_names.index(column)].tolist()
        live_avg.iloc[iter-1, column_names.index(column)] = np.float64(np.average(temp_list))

In [34]:
live_avg = live_avg.iloc[:-1,:]

In [36]:
from sklearn.preprocessing import normalize, StandardScaler

live_avg_norm = normalize(live_avg)
ss = StandardScaler()
live_avg_scaled = ss.fit_transform(live_avg)

In [38]:
live_avg.to_csv("live_avg.csv")